In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

INPUT_FOLDER = "/content/drive/MyDrive/traffic_video1"
OUTPUT_FOLDER = "/content/drive/MyDrive/traffic_output1"

# Check folders
print("Input exists:", os.path.exists(INPUT_FOLDER))
print("Output exists:", os.path.exists(OUTPUT_FOLDER))


Input exists: True
Output exists: False


In [3]:
# ================================
# Optimized Traffic Vehicle Detection + Speed + Risk Display + Total Traffic Risk
# ================================

# --- Step 1: Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from scipy.spatial import distance
import random

# --- Step 2: Set folders ---
INPUT_FOLDER = "/content/drive/MyDrive/traffic_video1"
OUTPUT_FOLDER = "/content/drive/MyDrive/trafficooo3"
os.makedirs(INPUT_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# --- Step 3: Video & Road parameters ---
ROAD_WIDTH_METERS = 10
VIDEO_FPS = 30
OVERSPEED_KMH = 50          # Speed limit threshold
FRAME_SKIP = 10             # Process every 10th frame for faster processing

RESIZE_WIDTH = 960          # Optional, to resize frames
RESIZE_HEIGHT = 540

# --- Step 4: Load SSD MobileNet ---
print("⏳ Loading SSD MobileNet model...")
detector = hub.load("https://tfhub.dev/tensorflow/ssd_mobilenet_v2/fpnlite_640x640/1")
print("✅ Model loaded!")

VEHICLE_CLASSES = {2: "bicycle", 3: "car", 4: "motorcycle", 6: "bus", 8: "truck"}
CONFIDENCE_THRESHOLD = 0.6

# --- Step 5: Vehicle tracker ---
class VehicleTracker:
    def __init__(self):
        self.next_id = 1
        self.vehicles = {}

    def update(self, centroids):
        updated = {}
        used_ids = set()
        for c in centroids:
            min_dist = float('inf')
            assigned_id = None
            for vid, prev_c in self.vehicles.items():
                if vid in used_ids:
                    continue
                d = distance.euclidean(c, prev_c)
                if d < min_dist and d < 50:
                    min_dist = d
                    assigned_id = vid
            if assigned_id is None:
                assigned_id = self.next_id
                self.next_id += 1
            updated[assigned_id] = c
            used_ids.add(assigned_id)
        self.vehicles = updated
        return updated

tracker = VehicleTracker()
prev_positions = {}

# Dictionary to store persistent vehicle risk states
vehicle_states = {}

# --- Step 6: Risk calculation ---
def calculate_risk(vehicle):
    risk = 0
    if vehicle["class_name"] in ["motorcycle", "bicycle"] and not vehicle.get("helmet", True):
        risk += 30
    if vehicle.get("zigzag", False):
        risk += 25
    if vehicle["class_name"] == "car" and not vehicle.get("seatbelt", True):
        risk += 20
    if vehicle.get("overspeed", False):
        risk += 40
    if vehicle.get("red_light", False):
        risk += 50
    return min(risk, 100)

# --- Step 7: TensorFlow detection function ---
@tf.function
def detect_tf(image_tensor):
    return detector(image_tensor)

# --- Step 8: Detection + display ---
def detect_and_display(frame):
    h, w, _ = frame.shape
    meters_per_pixel = ROAD_WIDTH_METERS / w

    input_tensor = tf.convert_to_tensor(frame, dtype=tf.uint8)
    input_tensor = tf.expand_dims(input_tensor, 0)
    results = detect_tf(input_tensor)

    boxes = results["detection_boxes"][0].numpy()
    scores = results["detection_scores"][0].numpy()
    classes = results["detection_classes"][0].numpy().astype(int)

    centroids = []
    valid_detections = []

    for i in range(len(scores)):
        if scores[i] < CONFIDENCE_THRESHOLD:
            continue
        class_id = classes[i]
        class_name = VEHICLE_CLASSES.get(class_id)
        if class_name is None:
            continue

        ymin, xmin, ymax, xmax = boxes[i]
        cx = int((xmin + xmax) / 2 * w)
        cy = int((ymin + ymax) / 2 * h)
        centroids.append((cx, cy))
        valid_detections.append((class_name, (xmin, ymin, xmax, ymax), cx, cy))

    tracked = tracker.update(centroids)
    total_traffic_risk = 0

    for vid, centroid in tracked.items():
        closest_det = min(valid_detections, key=lambda d: distance.euclidean(centroid, (d[2], d[3])))
        class_name, (xmin, ymin, xmax, ymax), cx, cy = closest_det

        prev_c = prev_positions.get(vid, centroid)
        prev_positions[vid] = centroid

        dist_m = distance.euclidean(centroid, prev_c) * meters_per_pixel
        speed_kmh = dist_m * VIDEO_FPS * 3.6

        # Persist risk attributes per vehicle ID
        if vid not in vehicle_states:
            vehicle_states[vid] = {
                "seatbelt": random.choice([True, False]) if class_name == "car" else True,
                "helmet": random.choice([True, False]) if class_name in ["motorcycle", "bicycle"] else True,
                "zigzag": random.choice([False, False, True]),
                "red_light": random.choice([False, False, True, False])
            }

        vehicle = {
            "class_name": class_name,
            "overspeed": speed_kmh > OVERSPEED_KMH,
            "seatbelt": vehicle_states[vid]["seatbelt"],
            "helmet": vehicle_states[vid]["helmet"],
            "zigzag": vehicle_states[vid]["zigzag"],
            "red_light": vehicle_states[vid]["red_light"]
        }

        risk_score = calculate_risk(vehicle)
        total_traffic_risk += risk_score

        # Smaller padding for bounding box
        padding_w = 0
        padding_h = 0
        xmin = int(xmin * w) - padding_w
        xmax = int(xmax * w) + padding_w
        ymin = int(ymin * h) - padding_h
        ymax = int(ymax * h) + padding_h

        xmin = max(0, xmin)
        xmax = min(w-1, xmax)
        ymin = max(0, ymin)
        ymax = min(h-1, ymax)

        # Color logic: red if overspeed or high risk, else green
        if vehicle["overspeed"] or risk_score >= 60:
            color = (0, 0, 255)
        else:
            color = (0, 255, 0)

        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, 2)

        lines = [f"{class_name}", f"{speed_kmh:.1f} km/h", f"Risk:{risk_score}"]
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5
        thickness = 1

        text_heights = [cv2.getTextSize(line, font, font_scale, thickness)[0][1] for line in lines]
        total_text_height = sum(text_heights) + 5 * len(lines)
        cv2.rectangle(frame, (xmin, ymin - total_text_height - 5), (xmax, ymin), color, -1)

        y0 = ymin - total_text_height
        for i, line in enumerate(lines):
            y_text = y0 + sum(text_heights[:i]) + i*5 + text_heights[i]
            cv2.putText(frame, line, (xmin + 5, y_text), font, font_scale, (255, 255, 255), thickness)

    cv2.rectangle(frame, (10,10), (220,50), (0,0,0), -1)
    cv2.putText(frame, f"Total Traffic Risk: {int(total_traffic_risk)}", (15,35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    return frame

# --- Step 9: Process videos ---
for video_file in os.listdir(INPUT_FOLDER):
    input_path = os.path.join(INPUT_FOLDER, video_file)
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("❌ Could not open video:", video_file)
        continue

    ret, sample_frame = cap.read()
    if not ret:
        cap.release()
        continue

    out_width, out_height = sample_frame.shape[1], sample_frame.shape[0]
    output_path = os.path.join(OUTPUT_FOLDER, f"processed_{video_file}")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, VIDEO_FPS // FRAME_SKIP, (out_width, out_height))

    print("⚡ Processing video:", video_file)
    frame_id = 0
    while True:
        if frame_id == 0:
            frame = sample_frame
        else:
            ret, frame = cap.read()
        if not ret:
            break
        frame_id += 1
        if frame_id % FRAME_SKIP != 0:
            continue

        processed_frame = detect_and_display(frame)
        out.write(processed_frame)

    cap.release()
    out.release()
    print("✅ Saved processed video to", output_path)

print("✅ All videos processed!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Loading SSD MobileNet model...
✅ Model loaded!
⚡ Processing video: 1900-151662242_medium.mp4
✅ Saved processed video to /content/drive/MyDrive/trafficooo3/processed_1900-151662242_medium.mp4
✅ All videos processed!
